# Data Integrity Checks and Cleaning

In [ ]:
# Public packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import math
import os
import re

# Custom packages
from filter import FilterDF as fdf
from benchmarks import ParetoAnalysis as pa
from benchmarks import AccuracyCalculation as ac
from integrity_fixes import DataFixer as fix, DataExporter as exporter

Working directory

In [ ]:
print(os.getcwd())

Preemptively set new Pandas feature

In [ ]:
pd.options.mode.copy_on_write = True

Allow changes to imported Python files without reseting the kernel

In [ ]:
%load_ext autoreload
%autoreload 2

## Import Data

Load formatted data

In [ ]:
%store -r static_data_processed
%store -r sales_data_processed

NA Checking

In [ ]:
total_entries = 0
total_na = 0
total_duplicated = 0
for loc_id, df in sales_data_processed.items():
    num_entries = df.shape[0]
    num_na = df.query('item_name.isna()', engine='python').shape[0]
    num_duplicated = df[df.duplicated()].shape[0]
    total_entries += num_entries
    total_na += num_na
    total_duplicated += num_duplicated
    # print(loc_id, num_na)
print("Total entries: ", total_entries)
print("Total NA: ", total_na)
print("Total duplicated: ", total_duplicated)

Read data from parquet files

In [ ]:
# Check if the data is already imported
if 'static_data_processed' not in locals():

    # Retrieve filnames
    static_data_filenames = os.listdir(f"1_palate_data_parquet")

    # Initialize a dictionary for the four parquet files of "static reference data"
    static_data = {}
    for filename in tqdm(static_data_filenames):

        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"1_palate_data_parquet/{filename}")
            base_name = re.sub(r'\.parquet$', '', filename)
            static_data[base_name] = df

    # Rename and store
    static_data_processed = static_data.copy()
    %store static_data_processed


# Data already exists
else:
    static_data = static_data_processed.copy()

# Check if the data is already imported
if 'sales_data_processed' not in locals():

    # Retrieve filnames
    restaurant_sales_filenames = os.listdir(f"1_palate_data_parquet/orders_item_level")

    # Initialize a dictionary for the 30 parquet files of "restaurant sales data"
    sales_data = {}
    for filename in tqdm(restaurant_sales_filenames):
        
        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"1_palate_data_parquet/orders_item_level/{filename}")
            location_id = re.sub(r'\.parquet$', '', filename)
            sales_data[location_id] = df
    
    # Rename and store
    sales_data_processed = {}
    for loc_id, df in sales_data.items():
        sales_data_processed[loc_id] = df.copy()
    %store sales_data_processed

# Data already exists
else:
    sales_data = {}
    for loc_id, df in sales_data_processed.items():
        sales_data[loc_id] = df.copy()


# Rename data for ease of use
before_after_details = static_data['before_after_details'].copy() # Promotional items (30 rows)
customers = static_data['customers'].copy() # Specific customer information: for matching customers with orders
items_tagged = static_data['items_tagged'].copy() # Menu items for all restaurants: for matching plant-based labels with orders
locations = static_data['locations'].copy() # Restaurant details (30 rows)

# Save precleaned versions
before_after_details_pre = before_after_details.copy()
customers_pre = customers.copy()
items_tagged_pre = items_tagged.copy()
locations_pre = locations.copy()
sales_data_pre = {}
for location_id, df in sales_data.items():
    sales_data_pre[location_id] = df.copy()

## Data Integrity Checks

### Totals

Data counts for later comparisons

In [ ]:
sales_total_pre = 0
sales_totals_pre = []
for loc, df in sales_data_pre.items():
    sales_totals_pre.append([loc, df.shape[0]])
    sales_total_pre += df.shape[0]
print(f"Precleaning total menu items: {items_tagged_pre.shape[0]}")
print(f"Precleaning total sales entries: {sales_total_pre}")


### Static Menu Data

In [ ]:
# Check what happens if we make changes, without doing committing to them yet
items_tagged_testing = items_tagged.copy()

# Make lowercase and strip white spaces to realistically check duplicates
items_tagged_testing['item_name'] = (items_tagged_testing['item_name']
                                     .str.lower()
                                     .str.strip())

# Count duplicates by different primary keys
menu_id_duplicates = items_tagged_testing['id'].duplicated().sum()
menu_name_duplicates_count = items_tagged_testing[items_tagged[['location_id', 'item_name']].duplicated(keep=False)]['item_name'].nunique() # Number of items that have a duplicate
menu_name_duplicates_total = items_tagged_testing[['location_id', 'item_name']].duplicated().sum() # Total number of duplicates

# Determine if these duplicates have meaningful differences
number_of_labels = items_tagged_testing.groupby(['location_id', 'item_name'])['is_plant_based'].nunique()
items_multiple_labels = number_of_labels[1 < number_of_labels]

# Output
print(f"Before and after number of menu items: {items_tagged_pre.shape[0]}, {items_tagged_testing.shape[0]}")
print(f"ID duplicates: {menu_id_duplicates}, Unique items duplicated: {menu_name_duplicates_count}, Total number of duplicates: {menu_name_duplicates_total}")
print(f"Are there items with multiple labelings: {(1 < number_of_labels).any()}")
print(f"Number of duplicates with different labels: {items_multiple_labels.index.size}")

# Sort by 'is_plant_based' within the primary key grouping of ('location_id', 'item_name') to prioritize removing 'unsure' labels over 'yes' and 'no's
items_tagged_testing = (items_tagged_testing
                        .sort_values(['location_id', 'item_name', 'is_plant_based'], ascending=True, ignore_index=True) # Note: sorting by a column first is equivalent to grouping by it
                        .drop_duplicates(['location_id', 'item_name'], keep='first', ignore_index=True) # Drop duplicates with different labels <---- potentially removes information, so may want to revisit this later
                        .dropna(subset='item_name', ignore_index=True)) # Drop NaN just for checking what happens

# Once done, we can check if the ID columns truly acts like a primary key within 
number_of_ids = items_tagged_testing.groupby(['location_id', 'item_name'])['id'].nunique()
items_multiple_ids = number_of_ids[1 < number_of_ids]
print(f"IDs are now unique: {items_multiple_ids.empty}")


### Restaurant Sales Data

Sales duplicate checking

In [ ]:
# Prepare dicts for exact duplicate sales data and a summary
exact_duplicates_dict = {}
id_duplicates_dict = {}
exact_duplicates_summary_list = []

# Prepare dicts for duplicate item names with different unit prices data and a summary
distinct_prices_data_dict = {}
items_with_distinct_unit_prices_dict = {}
distinct_prices_summary_list = []

# Capitalization
capitalization_summary_list = []

# Fractional Quantities
fractional_quantities_dict = {}

# For every restaurant
for location_id, df in tqdm(sales_data.items()):

    fix_df = fix(df, location_id)

    # Exact duplicate rows
    exact_duplicates, id_duplicates, non_exact_id_duplicates, exact_duplicate_summary_row = fix_df.find_exact_duplicates(primary_key=['unique_id'])
    
    # Add/Append
    exact_duplicates_dict[location_id] = exact_duplicates
    id_duplicates_dict[location_id] = id_duplicates
    exact_duplicates_summary_list.append(exact_duplicate_summary_row)


    # Items with distinct unit prices
    distinct_unit_prices_data, item_name_list, pricing_discrepancy_summary_row = fix_df.find_pricing_discrepancies()
    
    # Add/Append
    distinct_prices_data_dict[location_id] = distinct_unit_prices_data
    items_with_distinct_unit_prices_dict[location_id] = item_name_list
    distinct_prices_summary_list.append(pricing_discrepancy_summary_row)


    # Items with distinct capitalizations
    capitalization_summary = fix_df.find_capitalization_duplicates()

    # Append
    capitalization_summary_list.append(capitalization_summary)


    # Restaurants with fractional quantities
    fractional_quantities = fix_df.find_fractional_quantities()

    # Add
    fractional_quantities_dict[location_id] = fractional_quantities

More sales duplicate checking

In [ ]:
# Function for custom groupby aggregation
def check_duplicates(series):
    # Return True if there are any duplicates, False otherwise
    return series.duplicated().any()

unique_items_in_order = sales_data['0RJH3FFPYBPEY'].groupby(['order_id'])['item_name'].agg(check_duplicates)

## Data Integrity Logs & Records

### Static Menu Data

In [ ]:
menu_duplicates = items_tagged[items_tagged[['location_id', 'item_name']].duplicated(keep=False)].sort_values(['location_id','item_name'])
exporter.export_csv(menu_duplicates, 'palate_data_issues/menu_duplicates.csv')

### Restaurant Sales Data

In [ ]:
# Create summary dataframes
exact_duplicates_summary = pd.DataFrame(exact_duplicates_summary_list)
distinct_prices_summary = pd.DataFrame(distinct_prices_summary_list)
capitalization_summary = pd.DataFrame(capitalization_summary_list)

# Print
print((exact_duplicates_summary['rows_delivered'].sum(), exact_duplicates_summary['unique_rows_received'].sum()))

# Export
exporter.export_csv(exact_duplicates_summary, 'palate_data_issues/exact_duplicates_summary.csv')
# exporter.export_list(exact_duplicates_list, 'palate_data_issues/exact_duplicates.xlsx') <------------ TOO LARGE
exporter.export_csv(distinct_prices_summary, 'palate_data_issues/distinct_prices_summary.csv')
# exporter.export_list(distinct_prices_data_list, 'palate_data_issues/distinct_prices.xlsx')
exporter.export_csv(fractional_quantities_dict['75WYSXR9QBK5M'], 'palate_data_issues/fractional_quantities.csv')
exporter.export_csv(capitalization_summary, 'palate_data_issues/capitalization_discrepancies.csv')

## Data Cleaning

### Static Menu Data and Restaurant Sales Data

In [ ]:
# Function to handle the conversion to string while keeping NaNs
def to_string_or_na(series):
    return series.where(series.isna(), series.astype(str))

### Static reference data


## Customer data

# Sort so that the first duplicate to be removed is standardized
customers = (customers
             .astype({'gender':'category'})
             .sort_values(['location_id', 'customer_id', 'gender', 'age'], ignore_index=True)
             .drop_duplicates(['location_id', 'customer_id'], ignore_index=True) # Drop duplicates, i.e. customers with same IDs within a restaurant
             .dropna(subset=['customer_id'], ignore_index=True)) # Drop NaNs by customer ID

## Promotional data

## Location data

## Menu data

# Sort by 'is_plant_based' within the primary key grouping of ('location_id', 'item_name') to prioritize removing 'unsure' labels over 'yes' and 'no's
items_tagged = (items_tagged
                .assign(item_type = lambda df: df['item_type'].pipe(to_string_or_na),
                        dish_category = lambda df: df['dish_category'].pipe(to_string_or_na),
                        brand = lambda df: df['brand'].pipe(to_string_or_na))
                .dropna(subset=['item_name'], ignore_index=True) # Drop NaN item, which amounts to 1 thing as seen much above
                .assign(item_name = (lambda df: df['item_name'] # Standardize text formatting
                                     .str.title() # Assume items of different capitalizations are the same
                                     .str.strip()
                                     .replace(to_replace='üëΩ Space', value='\uf8ffÜëω Space')),
                        item_type = (lambda df: df['item_type']
                                     .str.title()
                                     .str.strip()),
                        dish_category = (lambda df: df['dish_category']
                                     .str.title()
                                     .str.strip()),
                        brand = (lambda df: df['brand']
                                     .str.title()
                                     .str.strip()),
                        is_alcohol = np.where(lambda df: df['dish_category'] == 'Alcohol', 'Yes', 'No'))
                .sort_values(['location_id', 'item_name', 'is_plant_based'], ascending=True, ignore_index=True) # Note: sorting by a column first is equivalent to grouping by it) 
                .drop_duplicates(['location_id', 'item_name'], keep='first', ignore_index=True)) # Drop duplicates with different labels <---- potentially removes information, so may want to revisit this later

### Note: ID is unique within the menu data and is therefore better than a composite primary key, but it fails to be a primary key because it doesn't correspond with the sales data
# Remove irrelevant column
if 'id' in items_tagged.columns:

    # Retrieve the item ids within the sales data
    restaurant_item_ids = set()
    for location_id, df in sales_data.items():
        restaurant_item_ids.union(df['unique_id'].tolist())
        
    # IDs from menu data
    print(f'Intersection of item IDs between sales data and menu data: {set(items_tagged["id"].tolist()).intersection(restaurant_item_ids)}')

    # Useless because there is absolutely no intersection, so remove
    if 'id' in items_tagged.columns:
        items_tagged.drop(axis=1, labels=['id'], inplace=True)

    ### Note: however, we cannot remove unique ID because there are multiple orders sold under a single transaction with the other distinction being unique id
    # for location_id, df in tqdm(sales_data.items()):
    #     df.drop(axis=1, labels=['unique_id'], inplace=True)
    #     sales_data[location_id] = df

# Fix encoding errors
%store -r encodings_df
menu_to_sales_encoding_conversion = encodings_df.copy()
for row in menu_to_sales_encoding_conversion.iterrows():
    items_tagged['item_name'] = items_tagged['item_name'].replace(to_replace=row[1][0], value=row[1][1])

# Save for counts later
items_tagged_post = items_tagged.copy()


### Restaurant sales data

# Save the total before dropping NaNs
sales_total_post_dropna = 0

# Save the total after doing all data cleaning
sales_total_post = 0

# Loop through
for location_id, df in tqdm(sales_data.items()):

    # Standardize text formatting
    df_na_removed = (df
          .assign(item_name = (df['item_name']
                               .str.title() # Pretend items of different capitalizations are the same <-------- *Note to potentially further analyze later*
                               .str.strip()))
          .dropna(subset='item_name')) # Only drop items that have NaN in 'item_name' column
    # Do NOT reset index, as it is a time series that doesn't need to be contiguous

    # Add to total to count data for after NaNs have been removed
    sales_total_post_dropna += df_na_removed.shape[0]

    # Drop perfect duplicates and count
    df_duplicates_removed = df_na_removed.drop_duplicates() # Do NOT reset index, as it is a time series that doesn't need to be contiguous
    
    # Add total to count data for after duplicates have been removed
    sales_total_post += df_duplicates_removed.shape[0]

    # Save
    sales_data[location_id] = df_duplicates_removed

# Remove non food items
%store -r sales_to_remove
to_remove = sales_to_remove.copy()
sales_data['3AXDVZJYN9DRS'] = fdf(sales_data['3AXDVZJYN9DRS']).filter('item_name', to_remove[0], exclude=True)
sales_data['ED5J990H5VAZT'] = fdf(sales_data['ED5J990H5VAZT']).filter('item_name', to_remove[1], exclude=True)

### Relabeled Menu Data

Incorporate relabeled data, and remove the invalid labels

In [ ]:
# Import relabeled items
items_tagged_new = pd.read_excel("1.5_palate_data_excel_redone/top_items_tagged.xlsx")


# Sort by 'is_plant_based' within the primary key grouping of ('location_id', 'item_name') to prioritize removing 'unsure' labels over 'yes' and 'no's
items_tagged_new = (items_tagged_new
                    .assign(item_name = lambda df: df['item_name'].pipe(to_string_or_na),
                            item_type = lambda df: df['item_type'].pipe(to_string_or_na),
                            dish_category = lambda df: df['dish_category'].pipe(to_string_or_na),
                            brand = lambda df: df['brand'].pipe(to_string_or_na),
                            is_plant_based = lambda df: df['is_plant_based'].pipe(to_string_or_na))
                    
                    .dropna(subset=['item_name'], ignore_index=True) # Drop NaN item, which amounts to 1 thing as seen much above
                    .assign(item_name = (lambda df: df['item_name'] # Standardize text formatting
                                         .str.title() # Assume items of different capitalizations are the same
                                         .str.strip()
                                         .replace(to_replace='Üë? Space', value='\uf8ffÜëω Space')), # Fix one encoding issue
                            item_type = (lambda df: df['item_type']
                                         .str.title()
                                         .str.strip()),
                            dish_category = (lambda df: df['dish_category']
                                             .str.title()
                                             .str.strip()),
                            brand = (lambda df: df['brand']
                                     .str.title()
                                     .str.strip()),
                            is_plant_based = (lambda df: df['is_plant_based']
                                              .str.title() 
                                              .astype('category')),
                            is_alcohol = np.where(lambda df: df['dish_category'] == 'Alcohol', 'Yes', 'No'),
                            item_description = np.nan)
                    .sort_values(['location_id', 'item_name', 'is_plant_based'], ascending=True, ignore_index=True) # Note: sorting by a column first is equivalent to grouping by it) 
                    .drop_duplicates(['location_id', 'item_name'])
                    .reindex(columns = items_tagged.columns)) # Rearrange after adding description
%store items_tagged_new

# Merge the old items and the relabeled items and filter to the old items that weren't relabeled
old_and_new_items_tagged = pd.merge(items_tagged, items_tagged_new, 
                                    on=['location_id', 'item_name'], 
                                    how='outer', 
                                    suffixes=('_old', '_new'), 
                                    indicator=True) # The indicator let's us catalogue merge misses
double_columns = old_and_new_items_tagged.columns
not_relabeled_items_with_merge = old_and_new_items_tagged[old_and_new_items_tagged['_merge'] == 'left_only']
relabeled_items_with_merge = old_and_new_items_tagged[old_and_new_items_tagged['_merge'] == 'both']
not_relabeled_items = not_relabeled_items_with_merge[['location_id', 'item_name'] + double_columns[double_columns.str.endswith("_old")].tolist()] # Select the columns that hold the newly tagged data
relabeled_items = relabeled_items_with_merge[['location_id', 'item_name'] + double_columns[double_columns.str.endswith("_new")].tolist()] # Select the columns that hold the newly tagged data

# Merge successful with no merge issues; known by checking the following for merge issues
print(old_and_new_items_tagged['_merge'].value_counts())

# Combine the items that weren't relabeled with the newly relabeled items to 
menu_data = pd.concat([relabeled_items, not_relabeled_items], ignore_index=True)

# How many unique items must it have to be considered a useful category?
unique_items = 5
dish_category_counts = items_tagged['dish_category'].value_counts()
usable_categories = dish_category_counts[dish_category_counts > unique_items].index.tolist()

# Throw these smaller categories into an'Other' category
dish_categories_other = items_tagged['dish_category'].where(items_tagged['dish_category'].isin(usable_categories), 'Other')
items_tagged = items_tagged.assign(dish_category=dish_categories_other)

## Data Merging

Merging the sales and menu data

In [ ]:
# Initialize a dict for merged dataframes
sales_and_menu_data = {}
merge_value_counts = pd.Series([0,0,0], name='count', index=pd.Index(['left_only','both','right_only'], name='_merge'))
for location_id, df in tqdm(sales_data.items()):
    
    # Perform outer merge just to determine merge misses (importantly, duplicates in the menu data have been handled)
    merged_outer = pd.merge(df.reset_index(drop=True), items_tagged, 
                            on=['item_name', 'location_id'], 
                            how='left', 
                            indicator=True) # Reset index to retrieve 'created_at' timeseries before merge
    
    # Print merge misses
    merge_value_counts = merge_value_counts + merged_outer['_merge'].value_counts()

    # Perform the merge on both 'item_name' and 'location_id'
    merged = (merged_outer
              .query('_merge in ["both","left_only"]')
              .set_index('created_at')) # Set 'created_at' back as the index

    # # Remove failed merges due to missing data on 27 and encoding errors elsewhere <-------- *Note to potentially remove later*
    # merged = merged[~merged['is_plant_based'].isna()]

    # Store in the dictionary
    sales_and_menu_data[location_id] = merged
        
print(merge_value_counts)

### Totals (Again)

In [ ]:
merged_total_post = 0
merged_totals_post = []
for loc, df in sales_and_menu_data.items():
    merged_totals_post.append([loc, df.shape[0]])
    merged_total_post += df.shape[0]
print(f"Precleaning total menu items: {items_tagged_pre.shape[0]}, Postcleaning total menu items: {items_tagged_post.shape[0]}, With new items: {items_tagged.shape[0]}")
print(f"Precleaning total sales entries: {sales_total_pre}, Post dropna total sales entries: {sales_total_post_dropna}, Postcleaning total sales entries: {sales_total_post}")
print(f"Postmerge entries: {merged_total_post}")

## Export

In [ ]:
# Note: (most commpressed) gzip > zstd > snappy > lz4 (fastest compression)

# Save
static_data['items_tagged'] = items_tagged

# Static data
for name, df in tqdm(static_data.items()):

    # Write to parquet
    df.to_parquet(f"2_palate_data_parquet_cleaned/{name}.parquet", compression='zstd', index=True)

# Sales data
for loc_id, df in tqdm(sales_and_menu_data.items()):

    # Write to parquet
    df.to_parquet(f"2_palate_data_parquet_cleaned/orders_item_level/{loc_id}_sales_and_menu.parquet", compression='zstd', index=True)